In [1]:
import pandas as pd
import numpy as np
import os.path as op
from scipy import stats
from scipy.stats import chi2_contingency

## Load Data

In [2]:
# Load the table files to get subject IDs
drawn_table_fn = "./dset/group-drawn/habenula/sub-group_task-rest_desc-1S2StTesthabenula_table.txt"
avg_table_fn = "./dset/group-avg/habenula/sub-group_task-rest_desc-1S2StTesthabenula_table.txt"

# Load participants.tsv
participants_fn = "./dset/participants.tsv"
participants_df = pd.read_csv(participants_fn, sep="\t", low_memory=False)

print(f"Total participants in dataset: {len(participants_df)}")

Total participants in dataset: 2156


## Extract Subjects from Drawn Habenula Analysis

In [3]:
# Load drawn table
drawn_table_df = pd.read_csv(drawn_table_fn, sep="\t")
drawn_subjects = drawn_table_df["Subj"].tolist()

print(f"Subjects in drawn habenula analysis: {len(drawn_subjects)}")
print(f"First 10 subjects: {drawn_subjects[:10]}")

Subjects in drawn habenula analysis: 1479
First 10 subjects: ['sub-29013', 'sub-29099', 'sub-29477', 'sub-29731', 'sub-0050695', 'sub-29319', 'sub-29118', 'sub-0050032', 'sub-0051110', 'sub-29468']


## Extract Demographics for Drawn Subjects

In [4]:
# Filter participants to only those in drawn analysis
drawn_demo_df = participants_df[
    participants_df["participant_id"].isin(drawn_subjects)
].copy()

# Select only the columns of interest
columns_of_interest = [
    "participant_id",
    "DX_GROUP",
    "AGE_AT_SCAN",
    "SEX",
    "HANDEDNESS_CATEGORY",
    "DSM_IV_TR",
    "COMORBIDITY",
    "CURRENT_MED_STATUS",
]

drawn_demo_df = drawn_demo_df[columns_of_interest]

# Convert DX_GROUP to numeric and replace with labels
drawn_demo_df["DX_GROUP"] = pd.to_numeric(drawn_demo_df["DX_GROUP"], errors="coerce")
drawn_demo_df["DX_GROUP_label"] = drawn_demo_df["DX_GROUP"].map(
    {1: "ASD", 2: "TD"}
)

# Convert age to numeric
drawn_demo_df["AGE_AT_SCAN"] = pd.to_numeric(
    drawn_demo_df["AGE_AT_SCAN"], errors="coerce"
)

print(f"\nDemographics extracted for {len(drawn_demo_df)} subjects")
drawn_demo_df.head(10)


Demographics extracted for 1479 subjects


,participant_id,DX_GROUP,AGE_AT_SCAN,SEX,HANDEDNESS_CATEGORY,DSM_IV_TR,COMORBIDITY,CURRENT_MED_STATUS,DX_GROUP_label
2,sub-0050004,1,19.09,1,R,1.0,NaN,0,ASD
3,sub-0050005,1,13.73,2,R,1.0,NaN,1,ASD
4,sub-0050006,1,13.37,1,L,1.0,NaN,0,ASD
5,sub-0050007,1,17.78,1,R,1.0,NaN,0,ASD
6,sub-0050008,1,32.45,1,R,1.0,NaN,1,ASD
8,sub-0050010,1,35.20,1,L,1.0,NaN,0,ASD
9,sub-0050011,1,16.93,1,L,1.0,NaN,0,ASD
10,sub-0050012,1,21.48,1,R,1.0,NaN,0,ASD
11,sub-0050013,1,9.33,1,R,1.0,NaN,0,ASD
12,sub-0050014,1,14.20,1,R,1.0,NaN,1,ASD


In [5]:
# Check unique values in DSM_IV_TR before cleaning
print("Unique values in DSM_IV_TR before cleaning:")
print(drawn_demo_df["DSM_IV_TR"].unique())
print(f"\nValue counts:")
print(drawn_demo_df["DSM_IV_TR"].value_counts(dropna=False))

Unique values in DSM_IV_TR before cleaning:
[ 1.000e+00  0.000e+00 -9.999e+03  2.000e+00  3.000e+00        nan]

Value counts:
DSM_IV_TR
 NaN       710
 0.0       417
 1.0       222
 2.0        66
-9999.0     36
 3.0        28
Name: count, dtype: int64


In [6]:
# Replace -9999 and -9999.0 with NaN across the entire DataFrame
drawn_demo_df.replace([-9999, -9999.0, "-9999", "`"], np.nan, inplace=True)

# Replace values in "HANDEDNESS_CATEGORY"
drawn_demo_df["HANDEDNESS_CATEGORY"].replace(
    {
        "1": "R",
        "1.0": "R",
        "2": "L",
        "2.0": "L",
        "Mixed": "Ambi",
        "3": "Mixed",
        "3.0": "Mixed",
        "L->R": "Ambi",
    },
    inplace=True,
)

drawn_demo_df["HANDEDNESS_CATEGORY"] = (
    drawn_demo_df["HANDEDNESS_CATEGORY"]
    .replace({"1.0": 1, "2.0": 2, "3.0": 3})
    .fillna(drawn_demo_df["HANDEDNESS_CATEGORY"])
)

# Convert handedness to categorical labels
drawn_demo_df["HANDEDNESS_CATEGORY"].replace(
    {"R": 1, "L": 2, "Ambi": 3, "Mixed": 3}, inplace=True
)

# Final conversion back to readable labels
drawn_demo_df["HANDEDNESS_CATEGORY"].replace({1: "R", 2: "L", 3: "Ambi"}, inplace=True)

# Convert SEX to labels
drawn_demo_df["SEX"].replace({1: "M", 2: "F"}, inplace=True)

# Convert DSM_IV_TR to labels
drawn_demo_df["DSM_IV_TR"].replace(
    {0: "Control", 1: "Autism", 2: "Asperger's", 3: "PDD-NOS", 4: "Asperger's or PDD-NOS"}, inplace=True
)

# Convert CURRENT_MED_STATUS to labels
drawn_demo_df["CURRENT_MED_STATUS"].replace(
    {"0": 0, "1": 1, "0.0": 0, "1.0": 1, "'": np.nan}, inplace=True
)
drawn_demo_df["CURRENT_MED_STATUS"].replace({0: "No medication", 1: "Medication"}, inplace=True)

print("Demographics cleaned and recoded:")
print(drawn_demo_df.head(10))

Demographics cleaned and recoded:
   participant_id  DX_GROUP  AGE_AT_SCAN SEX HANDEDNESS_CATEGORY DSM_IV_TR  \
2     sub-0050004         1        19.09   M                   R    Autism   
3     sub-0050005         1        13.73   F                   R    Autism   
4     sub-0050006         1        13.37   M                   L    Autism   
5     sub-0050007         1        17.78   M                   R    Autism   
6     sub-0050008         1        32.45   M                   R    Autism   
8     sub-0050010         1        35.20   M                   L    Autism   
9     sub-0050011         1        16.93   M                   L    Autism   
10    sub-0050012         1        21.48   M                   R    Autism   
11    sub-0050013         1         9.33   M                   R    Autism   
12    sub-0050014         1        14.20   M                   R    Autism   

   COMORBIDITY CURRENT_MED_STATUS DX_GROUP_label  
2          NaN      No medication            ASD  
3    

/var/folders/x1/rzvtxb3x273_5fvyk8ftl70r0000gn/T/ipykernel_51196/1702300493.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  drawn_demo_df["HANDEDNESS_CATEGORY"].replace(
/var/folders/x1/rzvtxb3x273_5fvyk8ftl70r0000gn/T/ipykernel_51196/1702300493.py:26: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always

## Clean and Recode Demographic Variables

## Calculate Summary Statistics by Diagnosis Group

In [7]:
def summarize_demographics(df, group_col="DX_GROUP_label"):
    """Calculate summary statistics for each diagnosis group."""
    summary_lines = []
    summary_lines.append("=" * 80)
    summary_lines.append("DEMOGRAPHIC SUMMARY - DRAWN HABENULA SUBJECTS")
    summary_lines.append("=" * 80)
    summary_lines.append("")
    
    # Overall sample size
    summary_lines.append(f"Total Sample Size: {len(df)}")
    summary_lines.append("")
    
    # Overall age statistics
    overall_age_mean = df["AGE_AT_SCAN"].mean()
    overall_age_std = df["AGE_AT_SCAN"].std()
    summary_lines.append(f"Overall Age: {overall_age_mean:.2f} ({overall_age_std:.2f}) years [mean (SD)]")
    summary_lines.append("")
    
    # Group by diagnosis
    for group in ["ASD", "TD"]:
        group_df = df[df[group_col] == group].copy()
        n = len(group_df)
        
        summary_lines.append("-" * 80)
        summary_lines.append(f"{group} GROUP (n = {n})")
        summary_lines.append("-" * 80)
        summary_lines.append("")
        
        # Age: mean (SD)
        age_mean = group_df["AGE_AT_SCAN"].mean()
        age_std = group_df["AGE_AT_SCAN"].std()
        summary_lines.append(
            f"Age: {age_mean:.2f} ({age_std:.2f}) years [mean (SD)]"
        )
        summary_lines.append("")
        
        # Sex: counts only
        summary_lines.append("Sex:")
        sex_counts = group_df["SEX"].value_counts(dropna=False)
        for sex_val, count in sex_counts.items():
            sex_label = str(sex_val) if pd.notna(sex_val) else "Missing"
            summary_lines.append(f"  {sex_label}: {count}")
        summary_lines.append("")
        
        # Handedness: counts only
        summary_lines.append("Handedness:")
        hand_counts = group_df["HANDEDNESS_CATEGORY"].value_counts(dropna=False)
        for hand_val, count in hand_counts.items():
            hand_label = str(hand_val) if pd.notna(hand_val) else "Missing"
            summary_lines.append(f"  {hand_label}: {count}")
        summary_lines.append("")
        
        # DSM_IV_TR: counts only
        summary_lines.append("DSM-IV-TR Diagnosis:")
        dsm_counts = group_df["DSM_IV_TR"].value_counts(dropna=False)
        for dsm_val, count in dsm_counts.items():
            dsm_label = str(dsm_val) if pd.notna(dsm_val) else "Missing"
            summary_lines.append(f"  {dsm_label}: {count}")
        summary_lines.append("")
        
        # Comorbidity: counts only
        summary_lines.append("Comorbidity:")
        comor_counts = group_df["COMORBIDITY"].value_counts(dropna=False)
        for comor_val, count in comor_counts.items():
            comor_label = str(comor_val) if pd.notna(comor_val) else "Missing"
            summary_lines.append(f"  {comor_label}: {count}")
        summary_lines.append("")
        
        # Medication status: counts only
        summary_lines.append("Current Medication Status:")
        med_counts = group_df["CURRENT_MED_STATUS"].value_counts(dropna=False)
        for med_val, count in med_counts.items():
            med_label = str(med_val) if pd.notna(med_val) else "Missing"
            summary_lines.append(f"  {med_label}: {count}")
        summary_lines.append("")
        summary_lines.append("")
    
    summary_lines.append("=" * 80)
    
    return "\n".join(summary_lines)


# Generate summary
summary_text = summarize_demographics(drawn_demo_df)
print(summary_text)

DEMOGRAPHIC SUMMARY - DRAWN HABENULA SUBJECTS

Total Sample Size: 1479

Overall Age: 16.49 (8.15) years [mean (SD)]

--------------------------------------------------------------------------------
ASD GROUP (n = 661)
--------------------------------------------------------------------------------

Age: 16.68 (8.23) years [mean (SD)]

Sex:
  M: 572
  F: 89

Handedness:
  R: 450
  Missing: 133
  L: 55
  Ambi: 23

DSM-IV-TR Diagnosis:
  Missing: 333
  Autism: 222
  Asperger's: 66
  PDD-NOS: 28
  Control: 12

Comorbidity:
  Missing: 625
  ADHD Inattentive: 5
  Dysthymia: 3
  Mood Disorder NOS: 3
  ADHD NOS: 2
  Generalized Anxiety Disorder: 2
  Anxiety Disorder NOS; Mood Disorder NOS: 1
  Social Phobia; Specific Phobia: spiders: 1
  ADHD Combined: 1
  Dysthymia; Agoraphobia dx;: 1
  Specific Phobia: needles/shots: 1
  MDE (past); Dysthymic disorder: 1
  Disruptive disorder NOS: 1
  Specific Phobia: Butterflies: 1
  Generalized Anxiety Disorder; Specific phobia; Enuresis; Encopresis: 1
  A

In [8]:
def calculate_group_comparisons(df):
    """Calculate statistical tests comparing ASD vs TD groups."""
    
    asd_df = df[df["DX_GROUP_label"] == "ASD"].copy()
    td_df = df[df["DX_GROUP_label"] == "TD"].copy()
    
    results = []
    
    # Age: Independent samples t-test
    asd_age = asd_df["AGE_AT_SCAN"].dropna()
    td_age = td_df["AGE_AT_SCAN"].dropna()
    
    if len(asd_age) > 0 and len(td_age) > 0:
        t_stat, p_val = stats.ttest_ind(asd_age, td_age)
        results.append({
            "Variable": "Age",
            "Test": "Independent t-test",
            "Statistic": f"t = {t_stat:.3f}",
            "p-value": f"{p_val:.4f}",
            "Significant": "Yes" if p_val < 0.05 else "No"
        })
    
    # Sex: Chi-square test
    sex_crosstab = pd.crosstab(df["DX_GROUP_label"], df["SEX"])
    if sex_crosstab.shape[0] > 1 and sex_crosstab.shape[1] > 1:
        chi2, p_val, dof, expected = stats.chi2_contingency(sex_crosstab)
        results.append({
            "Variable": "Sex",
            "Test": "Chi-square",
            "Statistic": f"χ² = {chi2:.3f} (df={dof})",
            "p-value": f"{p_val:.4f}",
            "Significant": "Yes" if p_val < 0.05 else "No"
        })
    
    # Handedness: Chi-square test (excluding missing values)
    hand_df = df[df["HANDEDNESS_CATEGORY"].notna()].copy()
    if len(hand_df) > 0:
        hand_crosstab = pd.crosstab(hand_df["DX_GROUP_label"], hand_df["HANDEDNESS_CATEGORY"])
        if hand_crosstab.shape[0] > 1 and hand_crosstab.shape[1] > 1:
            chi2, p_val, dof, expected = stats.chi2_contingency(hand_crosstab)
            results.append({
                "Variable": "Handedness",
                "Test": "Chi-square",
                "Statistic": f"χ² = {chi2:.3f} (df={dof})",
                "p-value": f"{p_val:.4f}",
                "Significant": "Yes" if p_val < 0.05 else "No"
            })
    
    # DSM_IV_TR: Chi-square test (excluding missing values)
    dsm_df = df[df["DSM_IV_TR"].notna()].copy()
    if len(dsm_df) > 0:
        dsm_crosstab = pd.crosstab(dsm_df["DX_GROUP_label"], dsm_df["DSM_IV_TR"])
        if dsm_crosstab.shape[0] > 1 and dsm_crosstab.shape[1] > 1:
            chi2, p_val, dof, expected = stats.chi2_contingency(dsm_crosstab)
            results.append({
                "Variable": "DSM-IV-TR",
                "Test": "Chi-square",
                "Statistic": f"χ² = {chi2:.3f} (df={dof})",
                "p-value": f"{p_val:.4f}",
                "Significant": "Yes" if p_val < 0.05 else "No"
            })
    
    # Comorbidity: Chi-square test (excluding missing values)
    comor_df = df[df["COMORBIDITY"].notna()].copy()
    if len(comor_df) > 0:
        comor_crosstab = pd.crosstab(comor_df["DX_GROUP_label"], comor_df["COMORBIDITY"])
        if comor_crosstab.shape[0] > 1 and comor_crosstab.shape[1] > 1:
            chi2, p_val, dof, expected = stats.chi2_contingency(comor_crosstab)
            results.append({
                "Variable": "Comorbidity",
                "Test": "Chi-square",
                "Statistic": f"χ² = {chi2:.3f} (df={dof})",
                "p-value": f"{p_val:.4f}",
                "Significant": "Yes" if p_val < 0.05 else "No"
            })
    
    # Medication status: Chi-square test (excluding missing values)
    med_df = df[df["CURRENT_MED_STATUS"].notna()].copy()
    if len(med_df) > 0:
        med_crosstab = pd.crosstab(med_df["DX_GROUP_label"], med_df["CURRENT_MED_STATUS"])
        if med_crosstab.shape[0] > 1 and med_crosstab.shape[1] > 1:
            chi2, p_val, dof, expected = stats.chi2_contingency(med_crosstab)
            results.append({
                "Variable": "Medication Status",
                "Test": "Chi-square",
                "Statistic": f"χ² = {chi2:.3f} (df={dof})",
                "p-value": f"{p_val:.4f}",
                "Significant": "Yes" if p_val < 0.05 else "No"
            })
    
    return pd.DataFrame(results)


# Calculate statistical comparisons
stats_df = calculate_group_comparisons(drawn_demo_df)
print("\n" + "=" * 80)
print("STATISTICAL COMPARISONS: ASD vs TD")
print("=" * 80)
print(stats_df.to_string(index=False))
print("\nNote: p < 0.05 indicates statistically significant group difference")


STATISTICAL COMPARISONS: ASD vs TD
         Variable               Test           Statistic p-value Significant
              Age Independent t-test           t = 0.815  0.4150          No
              Sex         Chi-square  χ² = 31.286 (df=1)  0.0000         Yes
       Handedness         Chi-square   χ² = 9.903 (df=2)  0.0071         Yes
        DSM-IV-TR         Chi-square χ² = 685.861 (df=3)  0.0000         Yes
Medication Status         Chi-square χ² = 192.596 (df=1)  0.0000         Yes

Note: p < 0.05 indicates statistically significant group difference


## Statistical Comparisons Between Groups

## Save Results

In [9]:
# Save demographics dataframe
output_csv = "./dset/group-drawn/habenula/demographics_drawn.csv"
drawn_demo_df.to_csv(output_csv, index=False)
print(f"Demographics dataframe saved to: {output_csv}")

# Save summary statistics
output_txt = "./dset/group-drawn/habenula/demographics_drawn_summary.txt"
with open(output_txt, "w") as f:
    f.write(summary_text)
    f.write("\n\n")
    f.write("=" * 80 + "\n")
    f.write("STATISTICAL COMPARISONS: ASD vs TD\n")
    f.write("=" * 80 + "\n")
    f.write(stats_df.to_string(index=False))
    f.write("\n\nNote: p < 0.05 indicates statistically significant group difference\n")
print(f"Summary statistics saved to: {output_txt}")

# Save statistical comparisons as CSV
stats_csv = "./dset/group-drawn/habenula/demographics_drawn_statistics.csv"
stats_df.to_csv(stats_csv, index=False)
print(f"Statistical comparisons saved to: {stats_csv}")

Demographics dataframe saved to: ./dset/group-drawn/habenula/demographics_drawn.csv
Summary statistics saved to: ./dset/group-drawn/habenula/demographics_drawn_summary.txt
Statistical comparisons saved to: ./dset/group-drawn/habenula/demographics_drawn_statistics.csv


In [10]:
import re

# List of DX_GROUP values to iterate through
dx_group_values = ["ASD", "TD"]

# Create an empty DataFrame with columns for COMORBIDITY
comorb_df = pd.DataFrame(columns=["COMORBIDITY"])

# Loop through each DX_GROUP value
for dx_group in dx_group_values:
    # Filter the DataFrame for the current DX_GROUP value
    comorb_array = drawn_demo_df[drawn_demo_df["DX_GROUP_label"] == dx_group]["COMORBIDITY"].unique()
    comorb_list = comorb_array.tolist()
    comorb_split = []
    for item in comorb_list:
        if isinstance(item, str):
            comorb_split.extend(item.split(";"))

    # Remove leading and trailing spaces from each item in the list
    comorb_split = [value.strip() for value in comorb_split]

    # Create a dictionary to map keywords to specific values
    keyword_mapping = {
        "anxiety": "Anxiety",
        "phobia": "Phobia",
        "tic": "Tic Disorder",
        "dysth": "Dysthymia",
        "enuresis": "Enuresis",
        "depr": "Depression",
        "MDD": "Depression",
        "adhd": "ADHD",
        "bipolar": "Bipolar Disorder",
        "encopresis": "Encopresis",
        "schizo": "Schizophrenic Disorder",
        "GAD": "GAD",
        "mood": "Mood Disorder",
        "learn": "Nonverbal Learning Disorder",
        "dyslexia": "Developmental Dyslexia",
        "tourettes": "Tourettes Disorder",
        "disrupt": "Disruptive Disorder",
        "sensory": "Sensory Integration Disorder",
        "ODD": "ODD",
        "PTSD": "PTSD",
    }

    # Replace keywords with their corresponding values using case-insensitive search
    for i, value in enumerate(comorb_split):
        for keyword, mapped_value in keyword_mapping.items():
            if re.search(keyword, value, re.IGNORECASE):
                comorb_split[i] = mapped_value

    # Filter out comorbidities that are not in the keyword mapping
    comorb_split = [
        value for value in comorb_split if value in keyword_mapping.values()
    ]

    # Create a DataFrame with a named column
    comorb_splitdf = pd.DataFrame(comorb_split, columns=["COMORBIDITY"])

    # Count unique values and add them to the comorb_df
    comorb_count = comorb_splitdf["COMORBIDITY"].value_counts()

    column_name = f"Count ASD" if dx_group == "ASD" else f"Count TD"
    comorb_df_group = pd.DataFrame(comorb_count.reset_index())

    comorb_df_group.columns = ["COMORBIDITY", column_name]

    # merge new columns with the empty DataFrame
    if comorb_df.empty:
        comorb_df = comorb_df_group
    else:
        comorb_df = comorb_df.merge(comorb_df_group, on="COMORBIDITY", how="outer")

    comorb_df.loc["Total Counts"] = comorb_df.sum(
        numeric_only=True, axis=0, skipna=True
    )


comorb_df = comorb_df
print(comorb_df)
# Save DataFrame to the CSV file
"""csv_file_path = op.join(data_dir, "abide_comorb_cleaned.csv")
comorb_df.to_csv(csv_file_path, index=False)"""

                      COMORBIDITY  Count ASD  Count TD
0                            ADHD        7.0       NaN
1                         Anxiety        7.0       NaN
2                      Depression        1.0       NaN
3             Disruptive Disorder        1.0       NaN
4                       Dysthymia        3.0       NaN
5                      Encopresis        2.0       NaN
6                        Enuresis        2.0       NaN
7                             GAD        1.0       NaN
8                   Mood Disorder        3.0       NaN
9                             ODD        1.0       NaN
10                         Phobia       10.0       NaN
11                            NaN       38.0       NaN
Total Counts                  NaN       76.0       0.0


'csv_file_path = op.join(data_dir, "abide_comorb_cleaned.csv")\ncomorb_df.to_csv(csv_file_path, index=False)'

In [11]:
# Define replacement lists
adhd = [
    "ADHD",
    "Tic Disorder",
    "Nonverbal Learning Disorder",
    "Developmental Dyslexia",
    "Tourettes Disorder",
    "Sensory Integration Disorder",
]
anxiety = ["Phobia", "GAD", "Anxiety", "PTSD"]
mood = ["Dysthymia", "Depression", "Mood Disorder"]
disrupt = ["ODD", "Disruptive Disorder"]
remove = ["Enuresis", "Encopresis", "Bipolar Disorder", "Schizophrenic Disorder"]

# Define replacement dictionary
replacements = {
    "ADHD": "ADHD/Other ND",
    "Tic Disorder": "ADHD/Other ND",
    "Nonverbal Learning Disorder": "ADHD/Other ND",
    "Developmental Dyslexia": "ADHD/Other ND",
    "Tourettes Disorder": "ADHD/Other ND",
    "Sensory Integration Disorder": "ADHD/Other ND",
    "Phobia": "Anxiety",
    "GAD": "Anxiety",
    "Anxiety": "Anxiety",
    "PTSD": "Anxiety",
    "Dysthymia": "Mood Disorder",
    "Depression": "Mood Disorder",
    "Mood Disorder": "Mood Disorder",
    "ODD": "Disruptive",
    "Disruptive Disorder": "Disruptive",
    "Enuresis": None,
    "Encopresis": None,
    "Bipolar Disorder": None,
    "Schizophrenic Disorder": None,
}

# Replace values in the 'COMORBIDITY' column based on the dictionary
comorb_df["COMORBIDITY"].replace(replacements, inplace=True)

# Drop rows where 'COMORBIDITY' is None (corresponding to values in the 'remove' list)
comorb_df.dropna(subset=["COMORBIDITY"], inplace=True)

# Combine counts for the same comorbidity names
comorb_clean_combined = (
    comorb_df.groupby("COMORBIDITY")
    .agg({"Count ASD": "sum", "Count TD": "sum"})
    .reset_index()
)

# Print the updated DataFrame with combined counts
print(comorb_clean_combined)

# Save DataFrame to the CSV file
"""csv_file_path = op.join(data_dir, "abide_comorb_simplified.csv")
comorb_clean_combined.to_csv(csv_file_path, index=False)"""

     COMORBIDITY  Count ASD  Count TD
0  ADHD/Other ND        7.0       0.0
1        Anxiety       18.0       0.0
2     Disruptive        2.0       0.0
3  Mood Disorder        7.0       0.0


/var/folders/x1/rzvtxb3x273_5fvyk8ftl70r0000gn/T/ipykernel_51196/1124228437.py:39: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  comorb_df["COMORBIDITY"].replace(replacements, inplace=True)


'csv_file_path = op.join(data_dir, "abide_comorb_simplified.csv")\ncomorb_clean_combined.to_csv(csv_file_path, index=False)'

In [12]:
# Filter out comorbidity categories with zero counts in either group
comorb_filtered = comorb_clean_combined[
    (comorb_clean_combined["Count ASD"] > 0) & (comorb_clean_combined["Count TD"] > 0)
].copy()

print(f"Comorbidity categories included in chi-square test: {len(comorb_filtered)}")
print(comorb_filtered)

if len(comorb_filtered) > 0:
    # Create the contingency table
    contingency_table = pd.DataFrame(
        {"ASD": comorb_filtered["Count ASD"], "TD": comorb_filtered["Count TD"]}
    ).T  # Transpose to match the expected format

    # Perform the chi-square test
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)

    # Print the results
    print("\n=== Comorbidity Chi-square Test ===")
    print("Contingency Table:")
    print(contingency_table)
    print(f"\nChi-square statistic: {chi2:.3f}")
    print(f"p-value: {p_value:.4f}")
    print(f"Degrees of freedom: {dof}")
    print("\nExpected frequencies:")
    print(expected)
else:
    print("\nNo comorbidity categories with counts in both groups - cannot perform chi-square test")

Comorbidity categories included in chi-square test: 0
Empty DataFrame
Columns: [COMORBIDITY, Count ASD, Count TD]
Index: []

No comorbidity categories with counts in both groups - cannot perform chi-square test


## Optional: Compare with Average Habenula Subjects

In [13]:
# Load average table
avg_table_df = pd.read_csv(avg_table_fn, sep="\t")
avg_subjects = avg_table_df["Subj"].tolist()

print(f"Subjects in average habenula analysis: {len(avg_subjects)}")

# Check overlap
drawn_set = set(drawn_subjects)
avg_set = set(avg_subjects)

overlap = drawn_set.intersection(avg_set)
only_drawn = drawn_set - avg_set
only_avg = avg_set - drawn_set

print(f"\nOverlap between drawn and average: {len(overlap)} subjects")
print(f"Only in drawn: {len(only_drawn)} subjects")
print(f"Only in average: {len(only_avg)} subjects")

Subjects in average habenula analysis: 1584

Overlap between drawn and average: 1479 subjects
Only in drawn: 0 subjects
Only in average: 105 subjects
